In [ ]:
import numpy as np
seleccion_inicial = np.random.choice(100,5, replace=False)
t = np.random.rand(100)
print(t[seleccion_inicial])
print(len(seleccion_inicial))

[0.28474659 0.14517125 0.87784606 0.51267121 0.65396469]
5


# Introducción

El algoritmo K-means funciona de la siguiente manera:


1.   Se define el número de clusters (N) que se va a buscar
2.   Se seleccionan N datos aleatoriamente y se asignan como ubucaciones aleatorias para cada uno de los centroides de los clusters que se van a buscar
3.   Los datos se separan teniendo en cuenta el centroide más cercano a cada dato
4.   Se calculan nuevamente los centroides teniendo en cuenta los datos pertenecientes a cada cluster
5.   Se repiten los pasos 3 y 4 hasta que no haya cambios en la ubicación de los centroides

# Ejemplo seleccionando datos aleatoriamente

En este ejemplo de k-means se parte de unos centriodes iniciales seleccionados aleatoriamente entre los datos.

Se aplica el algoritmo para buscar diferentes cantidades de grupos, inciando en 2, y luego se calcula el Silhuete para cada caso.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import colormaps
import matplotlib.colors as mcolors
np.random.seed(300)
nclusters_creados = 5
ndatos = 1000
x = []
y = []
#************************************************************
# nclusters es el número de clusters que se busca
#************************************************************
for i in range(nclusters_creados):
    cx = np.random.rand() * 30 - 15
    cy = np.random.rand() * 30 - 15
    x.extend(np.random.randn(ndatos) + cx)
    y.extend(np.random.randn(ndatos) + cy)
x = np.array(x)
y = np.array(y)
resultados_silhouette = []
colores = colormaps["tab20"].colors#(tab20) #['r', 'g', 'b', 'c', 'm', 'y', 'k']

#print(colores.colors)

for nclusters in range(2,15):
    tiempo = time.ctime()
    archivopdf = 'K-means-EZ-'+str(nclusters)+'-grupos' + tiempo.replace(':', '-') + '.pdf'
    archivopdf = archivopdf.replace('\n', '').replace(' ', '-')
    with PdfPages(archivopdf) as pdf:
        plt.figure()
        plt.plot(x, y, '.g')
        np.random.seed(1970)
        seleccion_inicial = np.random.choice(len(x),nclusters, replace=False)
        ######################################################################
        # Centroides iniciales
        ######################################################################
        xc = x[seleccion_inicial]
        yc = y[seleccion_inicial]

        N = nclusters_creados * ndatos

        plt.plot(xc, yc, '.k',markersize=20)
        pdf.savefig()
        plt.close()
        err1 = 1
        while err1 > 1e-8:
            plt.figure()
            xc1 = xc.copy()
            yc1 = yc.copy()
            dist1 = np.zeros((nclusters, N))
            labels = -np.ones(len(x))
            for k1 in range(nclusters):
                dist1[k1, :] = np.sqrt((x - xc[k1])**2 + (y - yc[k1])**2)
            i = 0
            for k1 in range(nclusters):
                k3 = np.ones(N, dtype=bool)
                for k2 in range(nclusters):
                    if k1 != k2:
                        k3 = k3 & (dist1[k1, :] < dist1[k2, :])
                if np.sum(k3) > 0:
                    color = colores[i]
                    labels[k3] = i
                    i=i+1
                    xt = x[k3]
                    yt = y[k3]
                    xc[k1] = np.mean(xt)
                    yc[k1] = np.mean(yt)
                    plt.scatter(xt, yt, color=color,marker='.')
            err1 = np.max(np.sqrt((xc - xc1)**2 + (yc - yc1)**2))
            plt.plot(xc1, yc1, '.k',markersize=20)
            pdf.savefig()
            plt.close()
    from sklearn import metrics
    datos = np.column_stack((x,y))
    resultados_silhouette.append([nclusters,metrics.silhouette_score(datos, labels, metric='euclidean')])

In [2]:
resultados_silhouette

[[2, 0.6720631120220043],
 [3, 0.6101960636702873],
 [4, 0.6267764910780025],
 [5, 0.6741408640681069],
 [6, 0.5978489643983831],
 [7, 0.5327672591040223],
 [8, 0.4448309805279216],
 [9, 0.3764939171748599],
 [10, 0.37551141981562497],
 [11, 0.3132965250220596],
 [12, 0.32396820521798725],
 [13, 0.320803208099266],
 [14, 0.32311389567855037]]

# Verificando la convergencia

Para verificar la convergencia se repite la aplicación del algoritmo varias veces, partiendo de diferentes valores iniciales del los centroides y conservando aquellas soluciones que tengan un mejor indicador de desempeño.

https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation


